In [1]:
import os
from PIL import Image
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import matplotlib.pyplot as plt
import torch.nn.functional as F


In [2]:
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms


class OregairuDualDataset(Dataset):
    def __init__(
        self,
        root_dir: str,
        transform_global=None,
        transform_local=None,
        cache_dir: str = "/kaggle/temp/face_cache",
    ):
        """
        root_dir: Đường dẫn tới thư mục 'train' hoặc 'test'
        transform_global: Bộ biến đổi cho ảnh toàn cảnh (Global)
        transform_local: Bộ biến đổi cho ảnh khuôn mặt (Local)
        cache_dir: Thư mục tạm lưu trữ ảnh mặt đã crop để tối ưu tốc độ DataLoader
        """
        self.root_dir = root_dir
        self.transform_global = transform_global
        self.transform_local = transform_local
        self.cache_dir = cache_dir
        
        # Khởi tạo bộ cắt mặt
        self.cropper = AnimeFaceCropper(target_size=(224, 224))
        if self.cache_dir is not None:
            os.makedirs(self.cache_dir, exist_ok=True)

        self.image_paths = []
        self.labels = []

        # 1. Quét danh sách lớp cố định
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}

        # 2. Duyệt qua từng thư mục gom đường dẫn ảnh và nhãn
        for cls_name in self.classes:
            cls_folder = os.path.join(root_dir, cls_name)
            if not os.path.isdir(cls_folder):
                continue
            label_idx = self.class_to_idx[cls_name]

            for img_name in os.listdir(cls_folder):
                img_path = os.path.join(cls_folder, img_name)
                self.image_paths.append(img_path)
                self.labels.append(label_idx)

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        # 1. Đọc ảnh gốc (dùng cho nhánh Global)
        img_raw = Image.open(img_path).convert("RGB")

        # 2. Xử lý ảnh khuôn mặt (Local) có cơ chế Cache chống nghẽn CPU
        img_local_raw = None
        if self.cache_dir is not None:
            # Tạo tên file cache duy nhất theo tên ảnh và thư mục cha
            img_filename = os.path.basename(img_path)
            parent_folder = os.path.basename(os.path.dirname(img_path))
            cache_file = os.path.join(self.cache_dir, f"{parent_folder}_{img_filename}")

            if os.path.exists(cache_file):
                img_local_raw = Image.open(cache_file).convert("RGB")
            else:
                img_local_raw = self.cropper.crop(img_raw)
                img_local_raw.save(cache_file)
        else:
            img_local_raw = self.cropper.crop(img_raw)

        # 3. Áp dụng các phép biến đổi (Transforms) riêng cho từng nhánh
        img_global = img_raw
        img_local = img_local_raw

        if self.transform_global is not None:
            img_global = self.transform_global(img_global)

        if self.transform_local is not None:
            img_local = self.transform_local(img_local)

        # Trả về bộ ba đúng định dạng yêu cầu của ModelTrainer và MyModel
        return img_global, img_local, label

In [3]:
# Bộ biến đổi chuẩn hóa pixel theo phân phối của ImageNet cho EfficientNetB0
imagenet_norm = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

# 1. Dành cho tập TRAIN
train_global_transform = transforms.Compose([
    transforms.Resize((240, 240)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    imagenet_norm,
])

train_local_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    imagenet_norm,
])

# 2. Dành cho tập TEST / VALIDATION (Không crop/xoay ngẫu nhiên, giữ ảnh nguyên bản)
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    imagenet_norm,
])

In [4]:
import os
import urllib.request
import cv2
import numpy as np
from PIL import Image


class AnimeFaceCropper:

  def __init__(
      self,
      cascade_path: str = "lbpcascade_animeface.xml",
      target_size: tuple = (224, 224),
  ):
    self.target_size = target_size

    # 1. Tự động tải file cascade nếu chưa tồn tại trong môi trường Kaggle
    if not os.path.exists(cascade_path):
      cascade_url = "https://raw.githubusercontent.com/nagadomi/lbpcascade_animeface/master/lbpcascade_animeface.xml"
      urllib.request.urlretrieve(cascade_url, cascade_path)

    self.face_cascade = cv2.CascadeClassifier(cascade_path)

  def crop(self, pil_img: Image.Image) -> Image.Image:
    """Nhận ảnh PIL RGB và trả về ảnh PIL RGB cận cảnh khuôn mặt (target_size)."""
    img_np = np.array(pil_img)
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)

    # Dò tìm khuôn mặt anime với các ngưỡng lọc chuẩn
    faces = self.face_cascade.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=7, minSize=(50, 50)
    )

    if len(faces) > 0:
      # Sắp xếp lấy khuôn mặt có diện tích lớn nhất (w * h)
      faces = sorted(faces, key=lambda f: f[2] * f[3], reverse=True)
      x, y, w, h = faces[0]

      # Thêm viền đệm 20% để lấy trọn vẹn tóc và phụ kiện
      pad_x = int(w * 0.2)
      pad_y = int(h * 0.2)
      x1 = max(0, x - pad_x)
      y1 = max(0, y - pad_y)
      x2 = min(img_np.shape[1], x + w + pad_x)
      y2 = min(img_np.shape[0], y + h + pad_y)

      cropped = img_np[y1:y2, x1:x2]
    else:
      # Fallback: Cắt Top-Center (70% diện tích lệch lên phía trên)
      h_img, w_img = img_np.shape[:2]
      crop_size = int(min(h_img, w_img) * 0.7)
      start_y = int(h_img * 0.1)
      start_x = (w_img - crop_size) // 2

      if start_y + crop_size > h_img:
        start_y = h_img - crop_size

      cropped = img_np[
          start_y : start_y + crop_size, start_x : start_x + crop_size
      ]

    # Đưa về kích thước chuẩn và chuyển ngược lại dạng PIL
    cropped = cv2.resize(cropped, self.target_size)
    return Image.fromarray(cropped)

In [5]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class ArcMarginProduct(nn.Module):

  def __init__(
      self,
      in_features: int,
      out_features: int,
      s: float = 30.0,
      m: float = 0.30,
  ):
    super().__init__()
    self.in_features = in_features 
    self.out_features = out_features
    self.s = s # hệ số phóng đại s
    self.m = m # góc lệch m
    self.weight = nn.Parameter(torch.empty(out_features, in_features))
    nn.init.xavier_uniform_(self.weight)
    self.cos_m = math.cos(m)
    self.sin_m = math.sin(m)
    self.th = math.cos(math.pi - m)
    self.mm = math.sin(math.pi - m) * m

  def forward(
      self, x: torch.Tensor, labels: torch.Tensor = None
  ) -> torch.Tensor:
    x_norm = F.normalize(x, p=2, dim=1)
    w_norm = F.normalize(self.weight, p=2, dim=1)
    cos_theta = F.linear(x_norm, w_norm)
    cos_theta = torch.clamp(cos_theta, min=-1.0 + 1e-7, max=1.0 - 1e-7)
    if labels is None:
      return self.s * cos_theta
    sin_theta = torch.sqrt(torch.clamp(1.0 - cos_theta**2, min=1e-7))
    phi = cos_theta * self.cos_m - sin_theta * self.sin_m
    phi = torch.where(cos_theta > self.th, phi, cos_theta - self.mm)
    one_hot = F.one_hot(labels, num_classes=self.out_features).to(cos_theta.dtype)
    output = (one_hot * phi) + ((1.0 - one_hot) * cos_theta)
    logits = self.s * output
    return logits
        

In [6]:
import torch
import torch.nn as nn
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0
class MyModel(nn.Module):
    def __init__(self,
      num_classes: int = 3,
      embedding_dim: int = 512,
      s: float = 30.0,
      m: float = 0.3):
        super().__init__()
        base_model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        self.backbone=base_model.features
        # Lấy danh sách toàn bộ các module (Conv2d, BatchNorm2d, SiLU,...) bên trong backbone
        all_layers = [module for module in self.backbone.modules()]
        for layer in all_layers[:100]:
          for param in layer.parameters():
              param.requires_grad = False
        self.pool=nn.AdaptiveAvgPool2d((1,1))
        self.flatten=nn.Flatten()
        self.neck = nn.Sequential(nn.BatchNorm1d(num_features=1280*2),
                                  nn.Dropout(p=0.5),
                                  nn.Linear(1280*2,embedding_dim))
        self.arcface = ArcMarginProduct(in_features=embedding_dim,out_features=num_classes,s=s,m=m)
    def extract_branch_feature(self, x: torch.Tensor) -> torch.Tensor:
        x=self.backbone(x)
        x= self.pool(x)
        x= self.flatten(x)
        return x
    def forward(self,img_global:torch.Tensor,img_local:torch.Tensor,labels:torch.Tensor=None)->torch.Tensor:
        feat_global=self.extract_branch_feature(img_global)
        feat_local=self.extract_branch_feature(img_local)
        merged = torch.cat([feat_global, feat_local], dim=1)
        embeddings = self.neck(merged)
        logits=self.arcface(embeddings,labels) 
        
        return logits
    

In [7]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader


class ModelTrainer:

  def __init__(
      self,
      model: nn.Module,
      train_loader: DataLoader,
      val_loader: DataLoader,
      criterion: nn.Module = None,
      optimizer: torch.optim.Optimizer = None,
      device: torch.device = None,
      checkpoint_dir: str = "/kaggle/working",
  ):
    # TODO 1.1: Thiết lập self.device (nếu device is None thì tự detect cuda/cpu)
    self.device=(device if device is not None else ("cuda" if torch.cuda.is_available() else "cpu"))
    # và đẩy self.model lên self.device
    self.model=model.to(self.device)

    # TODO 1.2: Gán self.train_loader, self.val_loader, self.checkpoint_dir
    # Đảm bảo thư mục checkpoint_dir tồn tại (dùng os.makedirs)
    self.train_loader=train_loader
    self.val_loader=val_loader

    # TODO 1.3: Thiết lập self.criterion (mặc định CrossEntropyLoss)
    self.criterion=(criterion if criterion is not None else nn.CrossEntropyLoss())

    # TODO 1.4: Thiết lập self.optimizer
    # Nếu optimizer is None: lọc các tham số có requires_grad == True
    # rồi dùng torch.optim.Adam với lr=1e-4
    if optimizer is not None :
        self.optimizer=optimizer
    else:
        trainable_para=filter(lambda p:p.requires_grad,self.model.parameters())
        self.optimizer = torch.optim.Adam(trainable_para, lr=1e-4)
    self.checkpoint_dir=checkpoint_dir
    os.makedirs(self.checkpoint_dir,exist_ok=True)

    # TODO 1.5: Khởi tạo từ điển self.history để lưu 'train_loss', 'train_acc', 'val_loss', 'val_acc'
    self.history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
    }

  def train_epoch(self) -> tuple[float, float]:
    """Chạy 1 epoch huấn luyện."""
    # TODO 2.1: Bật chế độ huấn luyện cho model (model.train())
    self.model.train()
    # Khởi tạo các biến tích lũy: running_loss, correct, total
    running_loss=0
    correct=0
    total=0
    # TODO 2.2: Duyệt qua từng batch trong self.train_loader:
    for img_global,img_local,labels in self.train_loader:
        img_global=img_global.to(self.device,non_blocking=True)
        img_local=img_local.to(self.device,non_blocking=True)
        labels=labels.to(self.device,non_blocking=True)
    # Cấu trúc mỗi batch gồm: img_global, img_local, labels
    #  - Đẩy dữ liệu lên self.device (dùng non_blocking=True)
        self.optimizer.zero_grad()
        logits=self.model(img_global,img_local,labels)
        loss=self.criterion(logits,labels)
        loss.backward()
        self.optimizer.step()
    #  - Xóa gradient cũ (optimizer.zero_grad())
    #  - Chú ý quan trọng: Forward lúc Train BẮT BUỘC truyền labels vào model để kích hoạt góc phạt m của ArcFace!
    #    logits = self.model(img_global, img_local, labels)
    #  - Tính loss: criterion(logits, labels)
    #  - Lan truyền ngược (loss.backward()) và cập nhật (optimizer.step())
    #  - Tích lũy loss (.item() * batch_size) và số lượng đoán đúng (torch.argmax)
        running_loss+=loss.item()*img_global.size(0)
        preds=torch.argmax(logits,dim=1)
        correct+=(preds==labels).sum().item()
        total+=img_global.size(0)
    # TODO 2.3: Tính epoch_loss và epoch_acc rồi return cặp (epoch_loss, epoch_acc)
    epoch_loss=running_loss/total
    epoch_acc=correct/total*100
    return epoch_loss,epoch_acc

  def evaluate(self) -> tuple[float, float]:
    """Chạy 1 epoch đánh giá."""
    # TODO 3.1: Bật chế độ đánh giá cho model (model.eval())
    self.model.eval() 
    running_loss=0
    correct=0
    total=0
      
    # Khởi tạo running_loss, correct, total

    # TODO 3.2: Dùng ngữ cảnh torch.no_grad():
    with torch.no_grad():
        for img_global,img_local,labels in self.val_loader:
            img_global=img_global.to(self.device,non_blocking=True)
            img_local=img_local.to(self.device,non_blocking=True)
            labels = labels.to(self.device, non_blocking=True)
            logits=self.model(img_global,img_local)
            loss=self.criterion(logits,labels)
            running_loss+=loss.item()*labels.size(0)
            total+=labels.size(0)
            preds=torch.argmax(logits,dim=1)
            correct+=(labels==preds).sum().item()
    val_loss=running_loss/total
    val_acc=correct/total*100
    # Duyệt qua từng batch trong self.val_loader:
    #  - Đẩy dữ liệu lên self.device
    #  - Chú ý quan trọng: Forward lúc Test KHÔNG truyền labels (labels=None mặc định)
    #    logits = self.model(img_global, img_local)
    #  - Tính loss để theo dõi: criterion(logits, labels)
    #  - Tích lũy loss và số lượng đoán đúng (torch.argmax)

    # TODO 3.3: Tính val_loss và val_acc rồi return cặp (val_loss, val_acc)
    return val_loss,val_acc

  def fit(self, epochs: int = 15) -> dict:
    """Vòng lặp chính quản lý train/val qua từng epoch và lưu Checkpoint tốt nhất."""
    best_val_acc = 0.0
    best_model_path = os.path.join(self.checkpoint_dir, "best_arcface_model.pth")

    # TODO 4.1: Vòng lặp for epoch in range(1, epochs + 1):
    #  - Gọi self.train_epoch() lấy train_loss, train_acc
    #  - Gọi self.evaluate() lấy val_loss, val_acc
    #  - Lưu các chỉ số vào self.history
    for epoch in range (1,epochs+1):
        train_loss,train_acc=self.train_epoch()
        val_loss,val_acc=self.evaluate()
        self.history["train_loss"].append(train_loss)
        self.history["train_acc"].append(train_acc)
        self.history["val_loss"].append(val_loss)
        self.history["val_acc"].append(val_acc)
        saved_flag = ""
        if val_acc>best_val_acc:
            best_val_acc=val_acc
            torch.save(self.model.state_dict(),best_model_path)
            saved_flag="Da luu mo hinh"
        print(
            f"Epoch [{epoch:2d}/{epochs}] | "
            f"Train Loss: {train_loss:.4f} - Acc: {train_acc:6.2f}% | "
            f"Val Loss: {val_loss:.4f} - Acc: {val_acc:6.2f}%{saved_flag}")
    print("-" * 75)
    print(f"✅ Hoàn thành! Best Val Acc đạt: {best_val_acc:.2f}%")
    print(f"📦 Checkpoint tốt nhất được lưu tại: {best_model_path}")
            
    
        

    # TODO 4.2: Cơ chế Checkpoint:
    #  - Nếu val_acc > best_val_acc: cập nhật best_val_acc
    #    và dùng torch.save(self.model.state_dict(), best_model_path)

    # TODO 4.3: In log tiến trình từng epoch ra màn hình

    # TODO 4.4: Kết thúc vòng lặp, return self.history
    return self.history

In [8]:
train_dir = "/kaggle/input/datasets/vxdwkhoa/oregairuclass2/classificationproject2/train"
test_dir = "/kaggle/input/datasets/vxdwkhoa/oregairuclass2/classificationproject2/test"

train_dataset = OregairuDualDataset(
    train_dir,
    transform_global=train_global_transform,
    transform_local=train_local_transform,
    cache_dir="/kaggle/temp/train_face_cache"
)

val_dataset = OregairuDualDataset(
    test_dir,
    transform_global=eval_transform,
    transform_local=eval_transform,
    cache_dir="/kaggle/temp/val_face_cache"
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

# Lấy 1 batch ra kiểm tra shape
batch_global, batch_local, batch_labels = next(iter(train_loader))
print("✅ Shape img_global :", batch_global.shape)  # Kỳ vọng: torch.Size([16, 3, 224, 224])
print("✅ Shape img_local  :", batch_local.shape)   # Kỳ vọng: torch.Size([16, 3, 224, 224])
print("✅ Shape labels     :", batch_labels.shape)  # Kỳ vọng: torch.Size([16])

✅ Shape img_global : torch.Size([16, 3, 224, 224])
✅ Shape img_local  : torch.Size([16, 3, 224, 224])
✅ Shape labels     : torch.Size([16])


In [9]:
model = MyModel(num_classes=3)
trainer = ModelTrainer(model, train_loader, val_loader)
history = trainer.fit(epochs=15)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 175MB/s]


Epoch [ 1/15] | Train Loss: 5.3859 - Acc:  22.10% | Val Loss: 0.3879 - Acc:  92.58%Da luu mo hinh
Epoch [ 2/15] | Train Loss: 3.1182 - Acc:  52.25% | Val Loss: 0.2203 - Acc:  94.88%Da luu mo hinh
Epoch [ 3/15] | Train Loss: 2.5871 - Acc:  62.42% | Val Loss: 0.2936 - Acc:  94.63%
Epoch [ 4/15] | Train Loss: 2.0850 - Acc:  67.55% | Val Loss: 0.2505 - Acc:  96.16%Da luu mo hinh
Epoch [ 5/15] | Train Loss: 2.2310 - Acc:  68.79% | Val Loss: 0.2837 - Acc:  95.40%
Epoch [ 6/15] | Train Loss: 1.8328 - Acc:  71.62% | Val Loss: 0.2135 - Acc:  95.65%
Epoch [ 7/15] | Train Loss: 2.1110 - Acc:  73.03% | Val Loss: 0.2028 - Acc:  95.65%
Epoch [ 8/15] | Train Loss: 1.6698 - Acc:  77.28% | Val Loss: 0.2258 - Acc:  96.68%Da luu mo hinh
Epoch [ 9/15] | Train Loss: 1.7098 - Acc:  75.86% | Val Loss: 0.2460 - Acc:  96.42%
Epoch [10/15] | Train Loss: 1.8728 - Acc:  76.83% | Val Loss: 0.2858 - Acc:  95.91%
Epoch [11/15] | Train Loss: 1.7972 - Acc:  77.45% | Val Loss: 0.2753 - Acc:  95.65%
Epoch [12/15] | Trai